# Expt 9: Text Classification and Text Generation using Transformers
**Ibrahim 24BAD035**

> Run on Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). If needed, first run: `!pip install -q transformers datasets scikit-learn accelerate`

## Part A: Dataset Preparation and Text Preprocessing

In [ ]:
# Student: Ibrahim 24BAD035
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

print("Student:", "Ibrahim 24BAD035")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

SEED = 31
# Load IMDb movie review dataset
dataset = load_dataset("stanfordnlp/imdb")
print(dataset)

In [ ]:
# Explore dataset
print("Train samples:", len(dataset["train"]))
print("Test samples :", len(dataset["test"]))
print("Features     :", dataset["train"].features)

print("\nSample entries:")
for i in range(3):
    ex = dataset["train"][i * 8000]
    label = "positive" if ex["label"] == 1 else "negative"
    print(f"[{label}] {ex['text'][:200]}...\n")

# Class distribution
train_counts = pd.Series(dataset["train"]["label"]).value_counts().sort_index()
test_counts  = pd.Series(dataset["test"]["label"]).value_counts().sort_index()
dist = pd.DataFrame({"train": train_counts, "test": test_counts})
dist.index = ["negative (0)", "positive (1)"]
print(dist)

dist.plot(kind="bar", figsize=(6, 4), title="IMDb Class Distribution")
plt.ylabel("Number of reviews")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Use a shuffled subset for faster training on Colab
TRAIN_SIZE, TEST_SIZE = 4000, 1000
train_ds = dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
test_ds  = dataset["test"].shuffle(seed=SEED).select(range(TEST_SIZE))
print("Subset sizes -> train:", len(train_ds), "| test:", len(test_ds))

# Load tokenizer for BERT
MODEL_NAME = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Example of tokenization
sample = train_ds[0]["text"]
enc = tokenizer(sample, truncation=True, padding="max_length", max_length=32)
print("\nSample tokens   :", tokenizer.convert_ids_to_tokens(enc["input_ids"])[:20])
print("Sample input_ids:", enc["input_ids"][:20])

In [ ]:
# Tokenize with padding and truncation, convert to torch format
MAX_LEN = 256

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN)

train_tok = train_ds.map(tokenize_fn, batched=True)
test_tok  = test_ds.map(tokenize_fn, batched=True)

train_tok = train_tok.rename_column("label", "labels")
test_tok  = test_tok.rename_column("label", "labels")
cols = ["input_ids", "token_type_ids", "attention_mask", "labels"]
train_tok.set_format("torch", columns=cols)
test_tok.set_format("torch", columns=cols)

print(train_tok)
print("input_ids shape of one sample:", train_tok[0]["input_ids"].shape)

# Batch-wise processing check
from torch.utils.data import DataLoader
loader = DataLoader(train_tok, batch_size=16, shuffle=True)
batch = next(iter(loader))
print("Batch shapes:", {k: tuple(v.shape) for k, v in batch.items()})

## Part B: Fine-Tuning BERT for Text Classification

In [ ]:
# Load pretrained BERT for sequence classification (2 classes)
from transformers import AutoModelForSequenceClassification

id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id
)
print("Number of parameters:", sum(p.numel() for p in bert_model.parameters()))

In [ ]:
# Training parameters and metrics
import inspect
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": accuracy_score(labels, preds),
            "precision": precision, "recall": recall, "f1": f1}

args_kwargs = dict(
    output_dir="bert_imdb_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="no",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)
# Argument name differs between transformers versions
eval_key = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters else "evaluation_strategy"
args_kwargs[eval_key] = "epoch"
training_args = TrainingArguments(**args_kwargs)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    compute_metrics=compute_metrics,
)

In [ ]:
# Fine-tune BERT
train_result = trainer.train()
print("Training finished. Training loss:", round(train_result.training_loss, 4))

In [ ]:
# Evaluate on the test set
eval_results = trainer.evaluate()
print("Evaluation results.")
for k in ["eval_loss", "eval_accuracy", "eval_precision", "eval_recall", "eval_f1"]:
    print(f"{k:15s}: {eval_results[k]:.4f}")

# Loss curve from the training log
log = pd.DataFrame(trainer.state.log_history)
train_log = log[log["loss"].notna()] if "loss" in log else pd.DataFrame()
if len(train_log):
    plt.figure(figsize=(7, 4))
    plt.plot(train_log["step"], train_log["loss"], marker="o")
    plt.xlabel("Step"); plt.ylabel("Training loss")
    plt.title("BERT Fine-tuning Loss")
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
# Test the model with new text inputs
def predict_sentiment(texts):
    bert_model.eval()
    inputs = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt").to(bert_model.device)
    with torch.no_grad():
        logits = bert_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    preds = probs.argmax(axis=1)
    return [(id2label[p], float(probs[i, p])) for i, p in enumerate(preds)]

new_texts = [
    "This movie was absolutely fantastic! Great acting and a brilliant story.",
    "A complete waste of time. The plot was boring and the acting was terrible.",
    "I loved the cinematography, but the ending was disappointing.",
    "One of the best films I have seen this year, highly recommended.",
    "I could not finish it; the dialogue was awful and the pacing was painfully slow.",
]
print("New text predictions\n")
for text, (label, conf) in zip(new_texts, predict_sentiment(new_texts)):
    print(f"Text      : {text}\nPredicted : {label} (confidence {conf:.3f})\n")

In [ ]:
# Save the fine-tuned model and tokenizer
bert_model.save_pretrained("bert_imdb_finetuned")
tokenizer.save_pretrained("bert_imdb_finetuned")
print("Saved fine-tuned model and tokenizer to ./bert_imdb_finetuned")

## Part C: Implementing a Sequence-to-Sequence Application using Transformers

In [ ]:
# Load T5-small model and tokenizer
from transformers import T5ForConditionalGeneration, AutoTokenizer

t5_name = "google-t5/t5-small"
t5_tokenizer = AutoTokenizer.from_pretrained(t5_name)
t5_model = T5ForConditionalGeneration.from_pretrained(t5_name).to(device)
t5_model.eval()
print("T5-small loaded. Parameters:", sum(p.numel() for p in t5_model.parameters()))

In [ ]:
# Helper function for generation
def t5_generate(text, task_prefix="summarize: ", max_length=60, min_length=10,
                num_beams=1, do_sample=False, temperature=1.0, top_k=50,
                no_repeat_ngram_size=0):
    # T5 needs a task prefix, e.g. "summarize: " or "translate English to German: "
    inputs = t5_tokenizer(task_prefix + text, return_tensors="pt", truncation=True, max_length=512).to(device)
    gen_kwargs = dict(max_length=max_length, min_length=min_length, num_beams=num_beams,
                      early_stopping=num_beams > 1, no_repeat_ngram_size=no_repeat_ngram_size)
    if do_sample:
        gen_kwargs.update(do_sample=True, temperature=temperature, top_k=top_k)
    with torch.no_grad():
        output_ids = t5_model.generate(**inputs, **gen_kwargs)
    return t5_tokenizer.decode(output_ids[0], skip_special_tokens=True)

article = (
    "Artificial intelligence is transforming many industries. In healthcare, AI models help doctors "
    "detect diseases from medical images faster and more accurately. In finance, they are used to detect "
    "fraud and automate trading decisions. In education, intelligent tutoring systems personalize learning "
    "for each student. However, experts warn that issues such as data privacy, bias and job displacement "
    "must be addressed so that the benefits of AI can be shared fairly across society."
)

print("INPUT TEXT \n", article)
print("\nSUMMARY (greedy, max_length=60):")
print(t5_generate(article, max_length=60))

In [ ]:
# Other tasks supported by T5: translation
sentences = [
    "The weather is beautiful today.",
    "I am studying artificial intelligence at college.",
    "Transformers have changed natural language processing.",
]
print("English -> German translation\n")
for s in sentences:
    out = t5_generate(s, task_prefix="translate English to German: ", max_length=60, min_length=1, num_beams=4)
    print("EN:", s)
    print("DE:", out, "\n")

print("English -> French translation\n")
for s in sentences:
    out = t5_generate(s, task_prefix="translate English to French: ", max_length=60, min_length=1, num_beams=4)
    print("EN:", s)
    print("FR:", out, "\n")

In [ ]:
# Experiment with generation settings
settings = {
    "Greedy, max_length=30":        dict(max_length=30, num_beams=1),
    "Greedy, max_length=80":        dict(max_length=80, num_beams=1),
    "Beam search (4), max_len=60":  dict(max_length=60, num_beams=4),
    "Beam (4) + no_repeat_ngram=2": dict(max_length=60, num_beams=4, no_repeat_ngram_size=2),
    "Sampling T=0.7, top_k=50":     dict(max_length=60, do_sample=True, temperature=0.7, top_k=50),
    "Sampling T=1.3, top_k=50":     dict(max_length=60, do_sample=True, temperature=1.3, top_k=50),
}

torch.manual_seed(SEED)
records = []
for name, params in settings.items():
    out = t5_generate(article, **params)
    records.append({"setting": name, "output": out, "word_count": len(out.split())})
    print(f"[{name}]\n{out}\n(words: {len(out.split())})\n")

settings_df = pd.DataFrame(records)
settings_df

In [ ]:
# NOTE: This cell is reconstructed from the output shown in the lab report
# (summaries of Sports / Technology / Education texts); the original code was not visible in the document.
# It also saves the generated outputs for further analysis.
texts = {
    "Sports": "The home team won the championship final 3-1 after a dramatic comeback in the second half. The captain scored twice and was named player of the match, while fans celebrated across the city late into the night.",
    "Technology": "The company unveiled its latest smartphone with an improved camera, a faster processor and a battery that lasts two days. Analysts expect strong sales because the price is lower than that of its main competitors.",
    "Education": "The university announced a new scholarship program to support students from rural areas. The program will cover tuition fees and provide laptops, and applications will open next month.",
}
rows = []
for topic, txt in texts.items():
    summ = t5_generate(txt, max_length=40, min_length=10, num_beams=4)
    rows.append({"topic": topic, "input": txt, "summary": summ})
    print(f"[{topic}]\nInput  : {txt}\nSummary: {summ}\n")

pd.DataFrame(rows).to_csv("t5_generated_outputs.csv", index=False)
settings_df.to_csv("t5_generation_settings.csv", index=False)
print("Saved generated outputs to CSV files.")

## Part D: Model Testing and Output Analysis

In [ ]:
# Test fine-tuned BERT on labelled positive/negative samples
test_samples = [
    ("An amazing, heartwarming film with outstanding performances.", "POSITIVE"),
    ("Brilliant direction and a gripping screenplay. I loved every minute.", "POSITIVE"),
    ("Superb soundtrack and wonderful characters, a must-watch.", "POSITIVE"),
    ("Dull, predictable and badly acted. I regret watching it.", "NEGATIVE"),
    ("The worst movie I have ever seen. Terrible script and no emotion.", "NEGATIVE"),
    ("Boring from start to finish, a total disaster.", "NEGATIVE"),
    ("The movie was not good at all.", "NEGATIVE"),
    ("It was not bad, actually quite enjoyable.", "POSITIVE"),
]
texts = [t for t, _ in test_samples]
actual = [a for _, a in test_samples]
preds = predict_sentiment(texts)

result_df = pd.DataFrame({
    "text": texts, "actual": actual,
    "predicted": [p[0] for p in preds],
    "confidence": [round(p[1], 3) for p in preds],
})
result_df["correct"] = result_df["actual"] == result_df["predicted"]
pd.set_option("display.max_colwidth", 80)
print("BERT sample testing")
display(result_df)
print("Sample accuracy:", result_df["correct"].mean())

In [ ]:
# Detailed metrics on the full test subset: report + confusion matrix
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

pred_output = trainer.predict(test_tok)
y_pred = np.argmax(pred_output.predictions, axis=-1)
y_true = pred_output.label_ids

print("Classification report.\n")
print(classification_report(y_true, y_pred, target_names=["NEGATIVE", "POSITIVE"], digits=4))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["NEGATIVE", "POSITIVE"]).plot(cmap="Blues")
plt.title("BERT Confusion Matrix")
plt.show()

# Misclassified examples (to study errors)
wrong_idx = np.where(y_pred != y_true)[0][:3]
print("Examples of misclassified reviews:")
for i in wrong_idx:
    print(f"- Actual: {id2label[int(y_true[i])]}, Predicted: {id2label[int(y_pred[i])]}")
    print("  ", test_ds[int(i)]["text"][:250], "...\n")

In [ ]:
# Compare T5 outputs under different settings (length and readability)
comparison = settings_df.copy()
comparison["output"] = comparison["output"].str.slice(0, 120)
print("T5 generation comparison.")
display(comparison)

plt.figure(figsize=(9, 4))
plt.barh(settings_df["setting"], settings_df["word_count"], color="steelblue")
plt.xlabel("Words in generated output")
plt.title("Output Length vs Generation Setting")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()